# Naive Bayes from Scratch

## Learning Objectives
- Understand generative classification
- Implement Naive Bayes without sklearn
- Clearly see the role of conditional independence
- Connect Naive Bayes to logistic regression and deep learning

In [2]:
import numpy as np

## Problem Setup

We consider a **classification problem**.

Given:
- Input feature vector $x = (x_1, \dots, x_d)$
- Class label $y \in \{0, 1, \dots, K-1\}$

Goal:
$$
\hat{y} = \arg\max_y P(y|x)
$$

Naive Bayes models:
$$
P(y|x) \propto P(y) P(x|y)
$$

## Conditional Independence Assumption

Naive Bayes assumes:
$$
P(x|y) = \prod_{i=1}^d P(x_i|y)
$$

This assumption is:
- Almost always false
- Computationally efficient
- Surprisingly effective in practice

In [3]:
# toy dataset
X = np.array([
    [1, 0, 1],
    [1, 1, 1],
    [0, 0, 1],
    [0, 1, 0],
    [1, 0, 0]
])

y = np.array([1, 1, 0, 0, 0])

In [5]:
def estimate_prior(y):
    classes, counts = np.unique(y, return_counts=True)
    priors = counts / len(y)
    return dict(zip(classes, priors))

In [6]:
priors = estimate_prior(y)
priors

{np.int64(0): np.float64(0.6), np.int64(1): np.float64(0.4)}

In [7]:
def estimate_likelihood(X, y):
    """
    Bernoulli Naive Bayes
    """
    classes = np.unique(y)
    n_features = X.shape[1]
    
    likelihood = {}
    
    for c in classes:
        X_c = X[y == c]
        likelihood[c] = (X_c.sum(axis=0) + 1) / (X_c.shape[0] + 2)
    
    return likelihood

In [8]:
likelihood = estimate_likelihood(X, y)
likelihood

{np.int64(0): array([0.4, 0.4, 0.4]), np.int64(1): array([0.75, 0.5 , 0.75])}

In [9]:
def predict_nb(X, priors, likelihood):
    predictions = []
    
    for x in X:
        log_probs = {}
        for c in priors:
            log_prob = np.log(priors[c])
            log_prob += np.sum(
                x * np.log(likelihood[c]) +
                (1 - x) * np.log(1 - likelihood[c])
            )
            log_probs[c] = log_prob
        
        predictions.append(max(log_probs, key=log_probs.get))
    
    return np.array(predictions)

In [10]:
preds = predict_nb(X, priors, likelihood)
preds, y

(array([1, 1, 0, 0, 0]), array([1, 1, 0, 0, 0]))

In [11]:
X_test = np.array([
    [0, 0, 0]
])
preds_test = predict_nb(X_test, priors, likelihood)
print(preds_test)

[0]


In [12]:
class NaiveBayes:
    """
    Bernoulli Naive Bayes classifier implemented from scratch.

    Assumptions:
    1. Features are binary (0 or 1)
    2. Conditional independence:
       Given the class label y, all features x_i are independent.
    """

    def __init__(self):
        # Log prior probability of each class: log P(y)
        self.class_log_prior = None
        
        # Log conditional probabilities: log P(x_i = 1 | y)
        self.feature_log_prob = None
        
        # All unique class labels
        self.classes = None

    def fit(self, X, y):
        """
        Train the Naive Bayes classifier.

        Parameters
        ----------
        X : array of shape (N, D)
            N samples, D binary features
        y : array of shape (N,)
            Class labels
        """

        # basic statistics
        N, D = X.shape
        self.classes = np.unique(y)
        C = len(self.classes)

        # compute class prior probabilities P(y)
        # - This step is pure counting.
        # - These is no optimization, no gradient descent.
        # Count how many samples belong to each class
        class_counts = np.zeros(C)
        
        for idx, c in enumerate(self.classes):
            class_counts[idx] = np.sum(y == c)
        
        # Prior probability P(y) = count(y) / N
        # We store log-probabilities for numerical stability
        self.class_log_prior = np.log(class_counts / N)

        # Step 3: Compute conditional probabilities P(x_i = 1 | y)
        # feature_counts[k, i]:
        # number of samples in class k where feature i equals 1
        feature_counts = np.zeros((C, D))

        for idx, c in enumerate(self.classes):
            # Select samples belonging to class c
            X_c = X[y == c]
            
            # Sum over samples to count feature occurrences
            feature_counts[idx] = np.sum(X_c, axis=0)

        # Apply Laplace smoothing:
        # P(x_i = 1 | y) = (count + 1) / (count(y) + 2)
        #
        # This avoids zero probabilities, which would
        # otherwise kill the entire class likelihood.
        self.feature_log_prob = np.log(
            (feature_counts + 1) / (class_counts[:, None] + 2)
        )


    def predict(self, X):
        """
        Predict class labels for given samples.

        Parameters
        ----------
        X : array of shape (N, D)

        Returns
        -------
        predictions : array of shape (N,)
        """
        N, D = X.shape
        log_probs = np.zeros((N, len(self.classes)))

        # compute log-posterior for each class
        for i in range(N):
            for k in range(len(self.classes)):
                # Start with log prior: log P(y)
                log_prob = self.class_log_prior[k]
                
                # TODO: should save original probability instead of log(P)
                # Add log likelihoods feature by feature
                for d in range(D):
                    if X[i, d] == 1:
                        # log P(x_i = 1 | y)
                        log_prob += self.feature_log_prob[k, d]
                    else:
                        # log P(x_i = 0 | y) = log(1 - P(x_i = 1 | y))
                        log_prob += np.log(
                            1 - np.exp(self.feature_log_prob[k, d])
                        )
                
                log_probs[i, k] = log_prob

        return self.classes[np.argmax(log_probs, axis=1)]

In [13]:
import numpy as np

# Toy dataset: 5 samples, 3 binary features
X_nb = np.array([
    [1, 0, 1],
    [1, 1, 1],
    [0, 0, 1],
    [0, 1, 0],
    [1, 0, 0]
])

# Labels
y_nb = np.array([1, 1, 0, 0, 0])

# Initialize and train
nb = NaiveBayes()
nb.fit(X_nb, y_nb)

# Make predictions
preds_nb = nb.predict(X_nb)
print("Predicted labels:", preds_nb)
print("True labels:     ", y_nb)

X_test = np.array(
    [[1, 1, 0]]
)
print("Predicted labels for test: ", nb.predict(X_test))


Predicted labels: [1 1 0 0 0]
True labels:      [1 1 0 0 0]
Predicted labels for test:  [0]


## Key Takeaways

- Naive Bayes is a **generative model**
- Learning = counting + normalization
- Prediction = summing log probabilities
- Strong assumptions, simple computation